# Explore Watershed Settings

Step 3b of the pipeline (see `README.md`). Re-cluster an **already-computed UMAP embedding** interactively under different watershed settings (`bins`, `gaussian_stddev`, `num_clusters`, `compactness`) without re-running UMAP. UMAP is the slow step (minutes on the full dataset); the watershed on top of it takes well under a second per setting, which is what makes interactive exploration practical.

- **Embedding:** made by `run_umap.py` (step 3a), one folder per run under `data/umap_results/`.
- **Watershed:** `WatershedConfig`, `run_watershed` and `assign_point_clusters` are defined in this notebook (moved from the old `run_umap_watershed.py`). Each cluster grows from a density peak, and boundaries fall in the valleys between peaks (fixed Sep 30; see README step 3b).
- **Feature values per point** (for the cluster × feature heatmap): reloaded from `data/features_export/` with `load_features` / `prepare_data` from `run_umap.py`.

**Before running**: point `RESULTS_DIR` below at a UMAP run folder. It needs `point_metadata.csv` and `config.json`. Folders from `run_umap.py` and from the older `run_umap_watershed.py` both work.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
from matplotlib.colors import ListedColormap, Normalize
from pathlib import Path
from ipywidgets import interact, FloatSlider, IntSlider
from astropy.convolution import Gaussian2DKernel, convolve_fft
from scipy.spatial.distance import pdist, squareform
from skimage.segmentation import watershed
from scipy.ndimage import center_of_mass
from run_umap import UMAPConfig, load_features, prepare_data
from umap_clusters import (
    WatershedConfig, run_watershed, assign_point_clusters,
    BEHAVIOR_FAMILY, FAMILY_ORDER, FAMILY_CMAP_NAME, behavior_family_palette,
    label_cluster_centroids, draw_cluster_outlines,
)
from export_features_csv import FEATURE_NAMES

2026-10-28 15:44:20.069984: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# watershed clustering (WatershedConfig, run_watershed, assign_point_clusters)
# is imported from umap_clusters.py

# --- plots ---

def plot_watershed_clusters(clusters, centers, histogram_x, histogram_y, smoothed_histogram, bins, out_path):
    """Saves the smoothed density next to the watershed label map."""
    fig = plt.figure(figsize=(20, 8))
    ax1 = fig.add_subplot(121)
    ax1.imshow(smoothed_histogram.T, origin="lower",
               extent=[histogram_x[0], histogram_x[-1], histogram_y[0], histogram_y[-1]], cmap="inferno")
    ax1.set_xlabel("UMAP Dim 1"); ax1.set_ylabel("UMAP Dim 2"); ax1.set_title("Point Density")

    ax2 = fig.add_subplot(122)
    ax2.set_ylim(0, bins)
    ax2.imshow(clusters)
    for i in range(len(centers)):
        if np.isfinite(centers[i]).all():  # a requested cluster index can end up with zero pixels -> NaN center
            ax2.text(centers[i, 1], centers[i, 0], str(i + 1), horizontalalignment="center", color="white")
    ax2.set_xlabel("Histogram bin (Dim 1)"); ax2.set_ylabel("Histogram bin (Dim 2)")
    ax2.set_title("Watershed Cluster Map")

    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_watershed_outlines(clusters, histogram_x, histogram_y, out_path):
    """Saves the watershed region boundaries in UMAP coordinates."""
    fig, ax = plt.subplots(figsize=(10, 8))
    for label_id in np.unique(clusters):
        mask = (clusters == label_id).astype(float)
        ax.contour(mask, levels=[0.5],
                   extent=[histogram_x[0], histogram_x[-1], histogram_y[0], histogram_y[-1]],
                   colors="black", linewidths=1.2)
    ax.set_xlabel("UMAP Dim 1"); ax.set_ylabel("UMAP Dim 2"); ax.set_title("Watershed Outlines")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

In [3]:
# --- point this at the run whose embedding you want to explore ---
from pipeline_paths import DATA_ROOT

RESULTS_DIR = DATA_ROOT / "umap_results" / "20261001_221104_nn50_md0.001_euclidean_2d"
meta = pd.read_csv(RESULTS_DIR / "point_metadata.csv")
embedding = meta[["umap_x", "umap_y"]].to_numpy()

print(f"{len(meta)} points loaded from {RESULTS_DIR.name}")

211080 points loaded from 20261001_221104_nn50_md0.001_euclidean_2d


In [ ]:
# --- reload the raw (z-scored) feature matrix in the same row order as
# meta/embedding above, so the cluster-feature heatmap below can look up
# actual feature values per point, not just umap_x/umap_y (point_metadata.csv
# saves the embedding, not the 33 features that went in).
#
# This re-derives the matrix from the feature CSVs rather than re-running
# UMAP: with the run's own (drop_unannotated, zscore, num_samples,
# random_seed) from config.json, load_features + prepare_data (run_umap.py)
# are deterministic and reproduce the exact same rows in the exact same
# order -- the asserts below check that. It only lines up if the feature
# export hasn't changed since that UMAP run. features_dir itself is NOT
# reused from config.json -- it records wherever that run's features_export
# folder was (e.g. the remote server's path), not this machine's -- so it's
# left at UMAPConfig's own local default instead.
with open(RESULTS_DIR / "config.json") as f:
    saved_config = json.load(f)

raw_cfg = UMAPConfig(
    drop_unannotated=saved_config["drop_unannotated"],
    zscore=saved_config["zscore"],
    num_samples=saved_config["num_samples"],
    random_seed=saved_config["random_seed"],
)
df_raw = load_features(raw_cfg)
# keep only the trials this run used (feature files added since the run, e.g. new animals, are
# left out before z-scoring, so the z-scores match the run's own)
df_raw = df_raw[df_raw["trial"].isin(meta["trial"].unique())]
data_raw, meta_check = prepare_data(df_raw, raw_cfg)

# line the reloaded rows up with the run's rows by (trial, frame), so the order
# the group folders were loaded in (the order of GROUPS) doesn't matter
assert len(data_raw) == len(meta), f"row count mismatch: reloaded {len(data_raw)}, meta has {len(meta)}"
reloaded_row = pd.Series(np.arange(len(meta_check)), index=pd.MultiIndex.from_frame(meta_check[["trial", "frame"]]))
assert reloaded_row.index.is_unique, "a (trial, frame) pair appears twice in the reloaded features"
row_order = reloaded_row.reindex(pd.MultiIndex.from_frame(meta[["trial", "frame"]])).to_numpy()
assert not np.isnan(row_order).any(), "some frames of this run are not in the reloaded features -- different feature files?"
row_order = row_order.astype(int)
data_raw, meta_check = data_raw[row_order], meta_check.iloc[row_order].reset_index(drop=True)
assert (meta_check["trial"].to_numpy() == meta["trial"].to_numpy()).all() and \
       (meta_check["frame"].to_numpy() == meta["frame"].to_numpy()).all(), "rows could not be aligned to this run"

print(f"reloaded {data_raw.shape[0]} x {data_raw.shape[1]} raw feature matrix, aligned to embedding/meta")

In [ ]:
# behavior colors by family (BEHAVIOR_FAMILY, behavior_family_palette) come from
# umap_clusters.py

# computed once, independent of watershed settings, so explore()'s behavior
# panel doesn't recompute it on every slider change
behaviors_present = sorted(meta["behavior"].dropna().unique())
ordered_behaviors, behavior_colors = behavior_family_palette(behaviors_present)
behavior_cmap = ListedColormap(behavior_colors)
behavior_codes = pd.Categorical(meta["behavior"], categories=ordered_behaviors).codes  # -1 = unannotated

print(f"{len(ordered_behaviors)} behaviors mapped to families")

In [ ]:
def cluster_cmap(n):
    """tab20 repeats every 20 -- fine for a quick exploratory view, but a
    high num_clusters setting will start reusing colors past 20."""
    base = plt.cm.tab20.colors
    return ListedColormap([base[i % len(base)] for i in range(n)])


# label_cluster_centroids and draw_cluster_outlines are imported from umap_clusters.py


def plot_cluster_feature_heatmap(point_clusters, data_raw, feature_names):
    """Clusters x 33-feature heatmap of mean z-scored features, rows
    hierarchically ordered. Near-identical (adjacent, low-distance-merging)
    rows mean two clusters occupy the same region of feature space despite
    being split apart in UMAP/watershed space -- a sign num_clusters (or
    compactness) is set too high for this data."""
    cluster_ids = np.unique(point_clusters[point_clusters > 0])
    cluster_means = pd.DataFrame(
        [data_raw[point_clusters == c].mean(axis=0) for c in cluster_ids],
        index=[f"cluster {c}" for c in cluster_ids], columns=feature_names,
    )

    g = sns.clustermap(
        cluster_means, cmap="RdBu_r", center=0,
        row_cluster=True, col_cluster=False, method="average", metric="euclidean",
        figsize=(14, max(4, 0.35 * len(cluster_ids) + 2)),
        dendrogram_ratio=(0.14, 0.01), cbar_kws={"label": "mean z-score"},
        xticklabels=True, yticklabels=True,
    )
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=7)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=8)
    g.fig.suptitle("Mean z-scored features per cluster, rows hierarchically ordered ", y=1.02)

    # numeric companion to the visual check: the closest pair of cluster rows
    # in feature space -- a very small distance relative to the typical
    # pairwise distance is the same "identical rows" signal as two adjacent
    # leaves merging at ~0 height in the dendrogram above
    dists = squareform(pdist(cluster_means.to_numpy(), metric="euclidean"))
    np.fill_diagonal(dists, np.inf)
    i, j = np.unravel_index(np.argmin(dists), dists.shape)
    typical = np.median(dists[np.isfinite(dists)])
    print(f"closest cluster pair: {cluster_means.index[i]} <-> {cluster_means.index[j]} "
          f"(distance {dists[i, j]:.2f}, vs. median pairwise distance {typical:.2f})")


def explore(bins, gaussian_stddev, num_clusters, compactness):
    cfg = WatershedConfig(
        bins=bins, gaussian_stddev=gaussian_stddev,
        num_clusters=num_clusters, compactness=compactness,
    )
    clusters, centers, histogram_x, histogram_y = run_watershed(embedding, cfg)
    point_clusters = assign_point_clusters(embedding, clusters, histogram_x, histogram_y, bins)

    n_actual = len(np.unique(clusters[clusters > 0]))
    cmap = cluster_cmap(max(num_clusters, 1))

    # density map for the middle panel -- same bins/gaussian_stddev as the
    # watershed itself (histogram2d with no explicit range is deterministic
    # given the same embedding+bins, so this lines up exactly with
    # histogram_x/histogram_y from run_watershed above), so the contour
    # lines drawn on top show exactly the segmentation these settings
    # actually produced on this density surface
    hist, hx, hy = np.histogram2d(embedding[:, 0], embedding[:, 1], bins=bins)
    kernel = Gaussian2DKernel(x_stddev=gaussian_stddev, y_stddev=gaussian_stddev)
    smoothed = convolve_fft(hist, kernel)
    smoothed = smoothed / smoothed.max()

    x_bins = np.clip(np.digitize(embedding[:, 0], hx) - 1, 0, bins - 1)
    y_bins = np.clip(np.digitize(embedding[:, 1], hy) - 1, 0, bins - 1)
    point_density = smoothed[x_bins, y_bins]

    fig = plt.figure(figsize=(24, 7))

    ax1 = fig.add_subplot(131)
    order = np.random.default_rng(0).permutation(len(embedding))  # shuffle so no cluster systematically draws on top
    ax1.scatter(embedding[order, 0], embedding[order, 1], c=point_clusters[order],
                cmap=cmap, vmin=1, vmax=num_clusters, s=2, alpha=0.5, linewidths=0)
    label_cluster_centroids(ax1, embedding, point_clusters, text_color="black", outline_color="white")
    ax1.set_xlabel("UMAP Dim 1"); ax1.set_ylabel("UMAP Dim 2")
    ax1.set_title(f"Points by Cluster  (requested {num_clusters}, got {n_actual} non-empty)")

    ax2 = fig.add_subplot(132)
    density_order = np.argsort(point_density)  # draw densest points last so they aren't buried
    ax2.scatter(embedding[density_order, 0], embedding[density_order, 1],
                c=point_density[density_order], cmap="inferno", norm=Normalize(0, 1),
                s=2, alpha=0.6, linewidths=0)
    draw_cluster_outlines(ax2, clusters, histogram_x, histogram_y, color="white")
    label_cluster_centroids(ax2, embedding, point_clusters, text_color="white", outline_color="black")
    ax2.set_xlim(embedding[:, 0].min(), embedding[:, 0].max())
    ax2.set_ylim(embedding[:, 1].min(), embedding[:, 1].max())
    ax2.set_facecolor("black")
    ax2.set_xlabel("UMAP Dim 1"); ax2.set_ylabel("UMAP Dim 2")
    ax2.set_title(f"Density with Cluster Boundaries  (bins={bins})")

    # third panel -- all data, colored by annotated behavior (family palette
    # from the cell above: blue=Prosocial, purple=Avoidance, red=Aggression,
    # gray=Asocial), with the same watershed boundaries as ax2 overlaid in
    # black (ax3's background is light, unlike ax2's, so the outline color
    # flips for contrast) -- this updates with the sliders just like ax1/ax2
    # since it's recomputed from this call's clusters/histogram_x/histogram_y,
    # letting you directly compare cluster edges against behavioral structure.
    ax3 = fig.add_subplot(133)
    unannotated = behavior_codes < 0
    if unannotated.any():
        ax3.scatter(embedding[unannotated, 0], embedding[unannotated, 1],
                    color="0.9", s=1.5, alpha=0.3, linewidths=0)
    annotated = ~unannotated
    behavior_order = np.random.default_rng(0).permutation(annotated.sum())
    ax3.scatter(embedding[annotated, 0][behavior_order], embedding[annotated, 1][behavior_order],
                c=behavior_codes[annotated][behavior_order], cmap=behavior_cmap,
                vmin=0, vmax=len(ordered_behaviors) - 1, s=2, alpha=0.6, linewidths=0)
    draw_cluster_outlines(ax3, clusters, histogram_x, histogram_y, color="black")
    label_cluster_centroids(ax3, embedding, point_clusters, text_color="black", outline_color="white")
    ax3.set_xlim(embedding[:, 0].min(), embedding[:, 0].max())
    ax3.set_ylim(embedding[:, 1].min(), embedding[:, 1].max())
    ax3.set_xlabel("UMAP Dim 1"); ax3.set_ylabel("UMAP Dim 2")
    ax3.set_title("Points by Behavior (family-colored)")

    handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=behavior_colors[i], markersize=6)
               for i in range(len(ordered_behaviors))]
    fig.legend(handles, ordered_behaviors, loc="center left", bbox_to_anchor=(1.0, 0.5),
               fontsize=7, frameon=False, title="Behavior")

    plt.tight_layout()

    # cluster size balance -- a few giant clusters + many tiny ones is a sign
    # these settings are producing one dominant blob rather than meaningful splits
    sizes = pd.Series(point_clusters).value_counts(normalize=True).sort_values(ascending=False)
    print(f"largest cluster holds {sizes.iloc[0]:.1%} of points; "
          f"smallest {len(sizes)} clusters hold {sizes.iloc[-1]:.2%} each (min)")

    plot_cluster_feature_heatmap(point_clusters, data_raw, FEATURE_NAMES)

In [ ]:
# sliders + two image widgets that are overwritten on every redraw. The figures
# are drawn off-screen and handed to the widgets as PNGs, so nothing depends on
# how VS Code captures plot output (interact() showed each plot twice, an
# Output widget showed none)
import io
import html
from contextlib import redirect_stdout
import ipywidgets as widgets
from IPython.display import display

sliders = {
    "bins": IntSlider(description="bins", min=50, max=300, step=10, value=200, continuous_update=False),
    "gaussian_stddev": FloatSlider(description="gaussian_stddev", min=0.5, max=6.0, step=0.25, value=1.5,
                                   continuous_update=False),
    "num_clusters": IntSlider(description="num_clusters", min=2, max=40, step=1, value=6, continuous_update=False),
    "compactness": FloatSlider(description="compactness", min=0.0, max=0.5, step=0.01, value=0.05,
                               continuous_update=False, readout_format=".2f"),
}
for slider in sliders.values():
    slider.style.description_width = "110px"
overview_image = widgets.Image(format="png", layout=widgets.Layout(width="100%"))
heatmap_image = widgets.Image(format="png", layout=widgets.Layout(width="100%"))
explore_text = widgets.HTML()


def figure_png(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=80, bbox_inches="tight")
    return buffer.getvalue()


def redraw(change=None):
    explore_text.value = "drawing..."
    plt.close("all")
    printed = io.StringIO()
    try:
        with redirect_stdout(printed):
            explore(**{name: slider.value for name, slider in sliders.items()})
        overview, heatmap = [plt.figure(number) for number in plt.get_fignums()]
        overview_image.value, heatmap_image.value = figure_png(overview), figure_png(heatmap)
        explore_text.value = f"<pre>{html.escape(printed.getvalue())}</pre>"
    except Exception as error:   # errors inside a slider callback are otherwise invisible
        explore_text.value = f"<pre>{html.escape(repr(error))}</pre>"
    finally:
        plt.close("all")         # nothing left open for the notebook to draw a second time


for slider in sliders.values():
    slider.observe(redraw, names="value")

display(widgets.VBox(list(sliders.values()) + [explore_text, overview_image, heatmap_image]))
redraw()


### Save the slider figure as a standalone HTML

The sliders above need a running kernel. This cell draws the same figure once for every combination of the settings listed below, saves the images in a folder, and writes an HTML page next to it whose sliders switch between them. It opens in any browser without Python; to share it, send the HTML together with its image folder (e.g. zipped). `bins` is held fixed. Each combination takes a few seconds, so the number of combinations sets the run time.


In [ ]:
import html
import io
import itertools
from contextlib import redirect_stdout

EXPORT_PATH = Path("figures/watershed_explorer.html")   # images go in a folder of the same name
EXPORT_BINS = 200
EXPORT_NUM_CLUSTERS = list(range(4, 21))                # slider 1
EXPORT_STDDEVS = [1.0, 1.5, 2.0, 2.5, 3.0]              # slider 2
EXPORT_COMPACTNESS = [0.01, 0.02, 0.03, 0.04, 0.05]     # slider 3
EXPORT_DPI = 60                                         # image resolution; higher = sharper but larger files

image_dir = EXPORT_PATH.with_suffix("")
image_dir.mkdir(parents=True, exist_ok=True)
combos = list(itertools.product(enumerate(EXPORT_NUM_CLUSTERS), enumerate(EXPORT_STDDEVS), enumerate(EXPORT_COMPACTNESS)))
texts = {}
for k, ((a, num_clusters), (b, stddev), (c, compactness)) in enumerate(combos, start=1):
    key = f"{a}_{b}_{c}"                                # slider positions
    plt.close("all")
    printed = io.StringIO()
    with redirect_stdout(printed):
        explore(bins=EXPORT_BINS, gaussian_stddev=stddev, num_clusters=num_clusters, compactness=compactness)
    overview, heatmap = [plt.figure(number) for number in plt.get_fignums()]
    overview.savefig(image_dir / f"overview_{key}.jpg", dpi=EXPORT_DPI, bbox_inches="tight", pil_kwargs={"quality": 85})
    heatmap.savefig(image_dir / f"heatmap_{key}.jpg", dpi=EXPORT_DPI, bbox_inches="tight", pil_kwargs={"quality": 85})
    texts[key] = printed.getvalue()
    plt.close("all")
    print(f"\r{k}/{len(combos)} drawn", end="")

page = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Watershed explorer</title>
<style>
  body {{ margin: 0; padding: 16px; background: #ffffff; color: #1d1d1f;
         font: 14px/1.4 -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }}
  h1 {{ font-size: 18px; margin: 0 0 4px; }}
  .note {{ color: #6e6e73; margin: 0 0 12px; }}
  .controls {{ display: flex; flex-wrap: wrap; gap: 8px 32px; margin-bottom: 12px; }}
  label {{ display: flex; align-items: center; gap: 8px; }}
  input[type=range] {{ width: 220px; }}
  .value {{ font-variant-numeric: tabular-nums; min-width: 2.5em; font-weight: 600; }}
  img {{ max-width: 100%; height: auto; display: block; margin-bottom: 12px; }}
  pre {{ white-space: pre-wrap; color: #6e6e73; border-top: 1px solid #d2d2d7; padding-top: 8px; }}
</style></head><body>
<h1>Watershed clusters on the UMAP embedding</h1>
<p class="note">Run {html.escape(RESULTS_DIR.name)}; bins = {EXPORT_BINS}.
Each setting was drawn in advance; the sliders switch between those images.</p>
<div class="controls">
  <label>num_clusters <input id="nc" type="range" min="0" max="{len(EXPORT_NUM_CLUSTERS) - 1}" step="1" value="0">
    <span class="value" id="nc_value"></span></label>
  <label>gaussian_stddev <input id="sd" type="range" min="0" max="{len(EXPORT_STDDEVS) - 1}" step="1" value="0">
    <span class="value" id="sd_value"></span></label>
  <label>compactness <input id="cp" type="range" min="0" max="{len(EXPORT_COMPACTNESS) - 1}" step="1" value="0">
    <span class="value" id="cp_value"></span></label>
</div>
<pre id="text"></pre>
<img id="overview" alt="clusters, density and behavior on the UMAP embedding">
<img id="heatmap" alt="mean z-scored features per cluster">
<script>
const FOLDER = {json.dumps(image_dir.name)};
const VALUES = {{nc: {json.dumps(EXPORT_NUM_CLUSTERS)}, sd: {json.dumps(EXPORT_STDDEVS)}, cp: {json.dumps(EXPORT_COMPACTNESS)}}};
const TEXTS = {json.dumps(texts)};
const sliders = ["nc", "sd", "cp"].map(id => document.getElementById(id));
function show() {{
  const key = sliders.map(s => s.value).join("_");
  sliders.forEach(s => document.getElementById(s.id + "_value").textContent = VALUES[s.id][s.value]);
  document.getElementById("overview").src = FOLDER + "/overview_" + key + ".jpg";
  document.getElementById("heatmap").src = FOLDER + "/heatmap_" + key + ".jpg";
  document.getElementById("text").textContent = TEXTS[key];
}}
sliders.forEach(s => s.addEventListener("input", show));
show();
</script></body></html>"""
EXPORT_PATH.write_text(page)
total_mb = sum(p.stat().st_size for p in image_dir.glob("*.jpg")) / 1e6
print(f"\nwrote {EXPORT_PATH} and {len(combos) * 2} images in {image_dir}/ ({total_mb:.0f} MB)")
